This notebook is the final agent pipeline orchestrator.

It connects : personas, memory, reasoning, debate, counterfactuals, retrieval, ranking, and explanations
into intelligent recommendations.

TODO: Eventually this notebook will contain:

Persona
→ Memory
→ Debate
→ Counterfactuals
→ Retrieval
→ Ranking
→ Recommendation
→ Explanation

In [4]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [5]:
import pandas as pd
import numpy as np

In [6]:
from src.recommender.retrieval import *

In [7]:
from src.recommender.multiturn import *

In [8]:
# STEP 4 - LOAD DATA

persona_df = pd.read_csv(
    "../data/processed/persona_profiles.csv"
)

businesses = pd.read_csv(
    "../data/processed/businesses.csv"
)

In [9]:
# 5 — SAMPLE TEST PERSONA
persona = persona_df.sample(1).iloc[0]

In [10]:
# STEP 6 — RETRIEVE CANDIDATES
candidates = retrieve_candidates(
    persona,
    businesses
)

candidates.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
113731,_ab50qdWOk0DdB6XOrBitw,Acme Oyster House,724 Iberville St,New Orleans,LA,70130,29.954273,-90.068965,4.0,7568,1,"{'RestaurantsTakeOut': 'True', 'Alcohol': ""'fu...","Live/Raw Food, Seafood, Restaurants, Cajun/Creole","{'Monday': '11:0-22:0', 'Thursday': '11:0-22:0..."
112552,ac1AeYqs8Z4_e2X5M3if2A,Oceana Grill,739 Conti St,New Orleans,LA,70130,29.956231,-90.067563,4.0,7400,1,"{'RestaurantsGoodForGroups': 'True', 'Restaura...","Restaurants, Seafood, Cajun/Creole, Breakfast ...","{'Monday': '8:0-1:0', 'Tuesday': '8:0-1:0', 'W..."
91757,GXFMD0Z4jEVZBCsbPf4CTQ,Hattie B’s Hot Chicken - Nashville,112 19th Ave S,Nashville,TN,37203,36.151387,-86.796603,4.5,6093,1,"{'RestaurantsGoodForGroups': 'True', 'Business...","American (Traditional), Chicken Shop, Southern...","{'Monday': '0:0-0:0', 'Tuesday': '11:0-16:0', ..."
143157,ytynqOUb3hjKeJfRj5Tshw,Reading Terminal Market,51 N 12th St,Philadelphia,PA,19107,39.953341,-75.158855,4.5,5721,1,"{'RestaurantsGoodForGroups': 'True', 'Restaura...","Candy Stores, Shopping, Department Stores, Fas...","{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
147081,oBNrLz4EDhiscSlbOl8uAw,Ruby Slipper - New Orleans,200 Magazine St,New Orleans,LA,70130,29.951025,-90.067394,4.5,5193,1,"{'NoiseLevel': ""'loud'"", 'Caters': 'False', 'B...","Restaurants, American (Traditional), American ...","{'Monday': '0:0-0:0', 'Tuesday': '7:30-14:0', ..."


In [11]:
from src.recommender.multiturn import (
    run_multiturn_recommendation_step
)

from src.memory.user_memory import (
    initialize_user_memory
)

memory = initialize_user_memory()

top_business = candidates.iloc[19]

result = (
    run_multiturn_recommendation_step(

        persona_row=persona,

        business_row=top_business,

        memory=memory,

        context_name="celebration",

        predicted_rating=5
    )
)

print(
    result[
        "recommendation_explanation"
    ]
)

print("\n")

print(
    result[
        "memory_summary"
    ]
)

Based on your Reactive Reviewer behavioral profile,
Datz appears to be a meaningful fit
for your current situation.







Celebration contexts typically increase your tolerance for premium experiences. 





This recommendation achieved a compatibility
score of 1.


{'top_favorites': [('unknown', 1)], 'top_dislikes': [], 'interaction_count': 1, 'current_emotional_state': 'positive'}


In [12]:
len(candidates)

20

In [13]:
from src.recommender.conversational_agent import (
    conversational_agent
)

In [14]:
response = conversational_agent(

    "I want a chill rooftop in Lagos for date night"
)

response

{'type': 'clarification',
 'response': 'Mainland or Island?',
 'memory': {'vibe': 'chill',
  'occasion': 'date night',
  'venue_type': 'rooftop'}}

In [15]:
response = conversational_agent(

    "Island"
)

response

{'type': 'recommendation',
 'response': 'Based on your date night vibe, here are some recommendations:',
 'recommendations': [{'name': 'Àkéta Rooftop',
   'description': 'Great ambience with affordable meals and rooftop vibes.'},
  {'name': 'RSVP Lagos',
   'description': 'Lively atmosphere with premium dining experience.'},
  {'name': 'The Sky Bar',
   'description': 'Luxury rooftop setting with beautiful night views.'}],
 'memory': {'vibe': 'chill',
  'occasion': 'date night',
  'venue_type': 'rooftop',
  'area': 'Island'}}